In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

print(os.path.exists(
    "/content/drive/MyDrive/MultilingualFakeNews"
))

True


In [4]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/MultilingualFakeNews"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root added.")

Project root added.


In [5]:
import os

BASE = "/content/drive/MyDrive/MultilingualFakeNews"

folders = [
    "src",
    "data",
    "models",
    "notebooks"
]

for folder in folders:
    os.makedirs(
        f"{BASE}/{folder}",
        exist_ok=True
    )

print("Project structure created.")

Project structure created.


In [6]:
import os

print(os.listdir(
    "/content/drive/MyDrive/MultilingualFakeNews"
))

['data', 'models', 'src', 'notebooks']


In [ ]:
init_code = ""

with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/__init__.py",
    "w"
) as f:
    f.write(init_code)

print("Created __init__.py")

Created __init__.py


In [ ]:
dataset_code = '''
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=384
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].flatten(),

            "attention_mask":
                encoding["attention_mask"].flatten(),

            "label":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }
'''

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/dataset.py",
    "w"
) as f:
    f.write(dataset_code)

print("dataset.py created")

dataset.py created


In [ ]:
model_code = '''
import torch
import torch.nn as nn

from transformers import AutoModel

class XLMRClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            "xlm-roberta-base"
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            768,
            2
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        embedding = outputs.last_hidden_state.mean(
            dim=1
        )

        embedding = self.dropout(
            embedding
        )

        logits = self.fc(
            embedding
        )

        return logits
'''

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/model.py",
    "w"
) as f:
    f.write(model_code)

print("model.py created")

model.py created


In [ ]:
train_code = '''
import torch

def train_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    return (
        total_loss / len(data_loader),
        correct / total
    )
'''

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/train.py",
    "w"
) as f:
    f.write(train_code)

print("train.py created")

train.py created


In [ ]:
evaluate_code = '''
import torch

def eval_epoch(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in data_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            labels = batch["label"].to(device)

            outputs = model(
                input_ids,
                attention_mask
            )

            loss = criterion(
                outputs,
                labels
            )

            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    accuracy = correct / total

    return (
        total_loss / len(data_loader),
        accuracy
    )
'''

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/evaluate.py",
    "w"
) as f:
    f.write(evaluate_code)

print("evaluate.py created")

evaluate.py created


In [ ]:
import os

print(
    os.path.exists(
        "/content/drive/MyDrive/MultilingualFakeNews/src/evaluate.py"
    )
)

True


In [ ]:
import sys

sys.path.append(
    "/content/drive/MyDrive/MultilingualFakeNews"
)

from src.evaluate import eval_epoch

print("Import successful")

Import successful


In [ ]:
from src.dataset import FakeNewsDataset
from src.model import XLMRClassifier
from src.evaluate import eval_epoch
from src.train import train_epoch

print("Imports successful")

Imports successful


CREATING A CONFIGURATION CODE TO SAVE GLOBAL VARIABLES

In [ ]:
config_code = '''
# Model Configuration

MODEL_NAME = "xlm-roberta-base"

MAX_LEN = 384

NUM_CLASSES = 2


# Training Configuration

BATCH_SIZE = 8

LEARNING_RATE = 2e-5

DROPOUT = 0.3

RANDOM_STATE = 42


# Labels

LABEL_MAP = {
    "Legit": 0,
    "Fake": 1
}

ID2LABEL = {
    0: "Legit",
    1: "Fake"
}
'''

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/config.py",
    "w"
) as f:
    f.write(config_code)

print("config.py created")

config.py created


In [ ]:
from src.config import *

print(MODEL_NAME)
print(MAX_LEN)
print(BATCH_SIZE)

xlm-roberta-base
384
8


In [7]:
preprocessing_code = '''
import re
import pandas as pd

def clean_text(text):
    """
    Basic text cleaning.
    """

    text = str(text)

    text = re.sub(r"http\\S+", "", text)

    text = re.sub(r"\\s+", " ", text)

    return text.strip()


def prepare_dataframe(df, label_map):
    """
    Prepare a dataframe for model input.

    Parameters
    ----------
    df : pandas.DataFrame

    label_map : dict

    Returns
    -------
    pandas.DataFrame
    """

    df = df.copy()

    df["Topic"] = df["Topic"].fillna("")

    df["News"] = df["News"].fillna("")

    df["text"] = (
        df["Topic"].astype(str)
        + " "
        + df["News"].astype(str)
    )

    df["text"] = df["text"].apply(clean_text)

    if "Label" in df.columns:
        df["target"] = df["Label"].map(label_map)

    return df
'''

In [8]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/preprocessing.py",
    "w"
) as f:
    f.write(preprocessing_code)

print("preprocessing.py created")

preprocessing.py created
